# MIMIC-IV — Data Loading & Cohort Construction




## 1. Imports & Config

In [2]:
import os
import pandas as pd
import numpy as np

PATH = r"mimic-iv-3.1"
BASE_DIR = PATH
HOSP_DIR = os.path.join(BASE_DIR, "hosp")
ICU_DIR  = os.path.join(BASE_DIR, "icu")

# Sanity check
print(os.path.exists(HOSP_DIR), os.path.exists(ICU_DIR))

True True


Getting rid of gzip

In [3]:
def csv_path(directory, filename):
    gz    = os.path.join(directory, filename + ".gz")
    plain = os.path.join(directory, filename)
    if os.path.exists(gz):
        return gz
    elif os.path.exists(plain):
        return plain
    raise FileNotFoundError(f"{filename} not found in {directory}")

---
## 2. Patients

One row per patient. Gives us gender and age

In [4]:
patients = pd.read_csv(
    csv_path(HOSP_DIR, "patients.csv"),
    usecols=["subject_id", "gender", "anchor_age", "dod"]
)

patients["gender"]    = patients["gender"].map({"M": 0, "F": 1})
patients["dod_known"] = patients["dod"].notna().astype(int)
patients = patients.drop(columns=["dod"])

print(f"{len(patients)} patients")
patients.head(3)

FileNotFoundError: patients.csv not found in mimic-iv-3.1/hosp

## Admissions data set process

Admissions contains a lot of essential information such as insurance category, race and the expiry binary (showing deaths in hospital). I have dropped Uknown, categorised insurance as a binary ( medicare and medicaid are federal plans for disadvantaged people: seniors, low income individuals and disabilities so Other would indicate private/ employer insurance or self paying meaning in general a wealthier patient.)

There is also elective vs emergency admissions, unsure wether to keep this but seems like a important feature. 

In [5]:
admissions = pd.read_csv(
    csv_path(HOSP_DIR, "admissions.csv"),
    usecols=["subject_id", "hadm_id", "admittime", "dischtime",
             "admission_type", "insurance", "marital_status",
             "race", "hospital_expire_flag"],
    parse_dates=["admittime", "dischtime"]
)

# This is HOSPITALE LOS not ICU
admissions["los_hospital_hrs"] = (
    (admissions["dischtime"] - admissions["admittime"])
    .dt.total_seconds() / 3600
).round(2)

# Emergency vs elective
emergency_types = {
    "AMBULATORY OBSERVATION", "DIRECT EMER.", "DIRECT OBSERVATION",
    "EU OBSERVATION", "EW EMER.", "OBSERVATION ADMIT", "URGENT"
}
admissions["is_emergency"] = admissions["admission_type"].isin(emergency_types).astype(int)


race_map = {
    "WHITE": "WHITE",
    "BLACK/AFRICAN AMERICAN": "BLACK",
    "HISPANIC/LATINO": "HISPANIC",
    "ASIAN": "ASIAN",
    "AMERICAN INDIAN/ALASKA NATIVE": "INDIGENOUS",
    "OTHER": "OTHER",
}
admissions["race_group"] = (
    admissions["race"].str.upper().str.strip()
    .map(race_map)
)
admissions = admissions[admissions["race_group"].notna()].reset_index(drop=True)

admissions["insurance_group"] = (
    admissions["insurance"]
    .map({"Medicare": "government", "Medicaid": "government"})
    .fillna("private")
)

admissions = admissions.drop(columns=["admittime", "dischtime", "admission_type", "race", "insurance"])

print(f"{len(admissions)} admissions | Mortality rate: {admissions['hospital_expire_flag'].mean():.2%}")
print("\nRace distribution:")
print(admissions["race_group"].value_counts())
admissions.head(3)

440864 admissions | Mortality rate: 1.92%

Race distribution:
race_group
WHITE         336538
BLACK          75482
OTHER          19788
ASIAN           7809
INDIGENOUS      1247
Name: count, dtype: int64


,subject_id,hadm_id,marital_status,hospital_expire_flag,los_hospital_hrs,is_emergency,race_group,insurance_group
0,10000032,22595853,WIDOWED,0,18.87,1,WHITE,government
1,10000032,22841357,WIDOWED,0,24.37,1,WHITE,government
2,10000032,25742920,WIDOWED,0,42.10,1,WHITE,government


---
## 4. ICU Stays

First ICU Stay only as if there is a second stay that means they survived the first one anyway and it also keeps the final df cleaner with each row representing a single patient admission rather than the same patient repeatedly

In [6]:
icustays = pd.read_csv(
    csv_path(ICU_DIR, "icustays.csv"),
    usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime", "los"],
    parse_dates=["intime", "outtime"]
)

icustays = icustays.sort_values("intime").groupby("hadm_id").first().reset_index()
icustays["long_icu_stay"] = (icustays["los"] > 3).astype(int)

print(f"{len(icustays)} ICU stays")
icustays.head(3)

85242 ICU stays


,hadm_id,subject_id,stay_id,intime,outtime,los,long_icu_stay
0,20000094,14046553,35605481,2150-03-02 15:19:31,2150-03-03 11:28:38,0.839664,0
1,20000147,14990224,30503572,2121-08-30 16:33:54,2121-08-31 21:29:49,1.205498,0
2,20000351,17913090,30593599,2145-06-13 20:10:27,2145-06-14 16:49:38,0.860544,0


---
## 5. Comorbidities (from ICD diagnosis codes)

Each admission has a list of ICD diagnosis codes (both ICD-9 and ICD-10 depending on year). We map these to 8 binary flags for common conditions that are well-known mortality predictors.

The raw table is long format (one row per diagnosis per admission) — we pivot it to one row per admission with binary columns.

In [7]:
comorbidity_map = {
    "250": "diabetes",    "E11": "diabetes",    "E10": "diabetes",
    "401": "hypertension", "I10": "hypertension",
    "428": "heart_failure", "I50": "heart_failure",
    "496": "copd",        "J44": "copd",
    "585": "ckd",         "N18": "ckd",
    "414": "cad",         "I25": "cad",
    "291": "alcohol",     "F10": "alcohol",
    "584": "aki",         "N17": "aki",
}

def map_code(code):
    for prefix, name in comorbidity_map.items():
        if str(code).startswith(prefix):
            return name
    return None

diagnoses = pd.read_csv(
    csv_path(HOSP_DIR, "diagnoses_icd.csv"),
    usecols=["hadm_id", "icd_code"]
)

diagnoses["icd_code"]    = diagnoses["icd_code"].str.strip().str.upper()
diagnoses["comorbidity"] = diagnoses["icd_code"].apply(map_code)
diagnoses = diagnoses[diagnoses["comorbidity"].notna()]

comorbidities = (
    diagnoses.groupby(["hadm_id", "comorbidity"]).size()
    .gt(0).astype(int)
    .unstack("comorbidity").fillna(0).astype(int)
    .reset_index()
)
comorbidities.columns = ["hadm_id"] + [f"cm_{c}" for c in comorbidities.columns[1:]]

print(f"{len(comorbidities)} admissions with comorbidity data")
comorbidities.head(3)


351611 admissions with comorbidity data


,hadm_id,cm_aki,cm_alcohol,cm_cad,cm_ckd,cm_copd,cm_diabetes,cm_heart_failure,cm_hypertension
0,20000019,1,0,0,0,0,1,0,1
1,20000024,0,0,0,0,0,0,0,1
2,20000034,0,0,0,1,0,1,0,0


---
## 6. Lab Results (first 24h of ICU stay)

`labevents` is a massive long-format table — millions of rows, one per lab test per patient per timepoint. We:
1. Filter to 14 clinically relevant labs (creatinine, lactate, glucose etc.)
2. Keep only measurements from the **first 24 hours** of ICU admission (no future leakage)
3. Pivot to one row per stay with mean values

The 24h window is standard in MIMIC mortality prediction literature.

In [8]:
LAB_ITEMS = {
    50912: "creatinine",
    50971: "potassium",
    50983: "sodium",
    50902: "chloride",
    50882: "bicarbonate",
    51006: "urea_nitrogen",
    51221: "hematocrit",
    51222: "hemoglobin",
    51265: "platelets",
    51301: "wbc",
    50868: "anion_gap",
    50931: "glucose",
    50813: "lactate",
    50820: "ph",
}

valid_hadm_ids = set(icustays["hadm_id"].unique())
chunks = []

for chunk in pd.read_csv(
    csv_path(HOSP_DIR, "labevents.csv"),
    usecols=["hadm_id", "itemid", "charttime", "valuenum"],
    parse_dates=["charttime"],
    chunksize=500_000,
    low_memory=False
):
    chunk = chunk[
        chunk["itemid"].isin(LAB_ITEMS) &
        chunk["hadm_id"].isin(valid_hadm_ids) &
        chunk["valuenum"].notna()
    ].copy()
    chunks.append(chunk)

labevents = pd.concat(chunks, ignore_index=True)
labevents["lab_name"] = labevents["itemid"].map(LAB_ITEMS)
labevents = labevents.merge(icustays[["hadm_id", "stay_id", "intime"]], on="hadm_id")
labevents = labevents[
    (labevents["charttime"] >= labevents["intime"]) &
    (labevents["charttime"] <= labevents["intime"] + pd.Timedelta(hours=24))
]

labs = (
    labevents.groupby(["stay_id", "lab_name"])["valuenum"]
    .mean().unstack("lab_name").reset_index()
)
labs.columns = ["stay_id"] + [f"lab_{c}" for c in labs.columns[1:]]

print(f"{len(labs)} stays with lab data, {labs.shape[1]-1} lab features")
labs.head(3)

83279 stays with lab data, 14 lab features


,stay_id,lab_anion_gap,lab_bicarbonate,lab_chloride,lab_creatinine,lab_glucose,lab_hematocrit,lab_hemoglobin,lab_lactate,lab_ph,lab_platelets,lab_potassium,lab_sodium,lab_urea_nitrogen,lab_wbc
0,30000153,12.000000,21.000000,115.000000,1.000000,168.000000,31.6,10.3,1.7,7.303333,167.5,4.600000,143.500000,22.000000,16.1
1,30000213,15.666667,23.333333,100.666667,3.666667,214.333333,23.9,7.4,0.9,7.405000,226.0,4.766667,139.666667,60.666667,5.8
2,30000484,10.000000,27.000000,104.000000,1.200000,94.000000,24.6,8.1,1.8,7.330000,357.0,5.300000,136.000000,47.000000,24.2


---
## 7. Vitals / Chart Events (first 24h of ICU stay)

`chartevents` is the largest table — continuous bedside monitoring data. Same approach as labs: filter to our vital signs of interest, restrict to the first 24h, and pivot wide.

We also compute `gcs_total` (Glasgow Coma Scale) by summing the three component scores — it's a standard neurological severity measure.

In [9]:
CHART_ITEMS = {
    220045: "heart_rate",
    220179: "sbp",
    220180: "dbp",
    220181: "map",
    220210: "resp_rate",
    220277: "spo2",
    223761: "temperature_f",
    220739: "gcs_eye",
    223900: "gcs_verbal",
    223901: "gcs_motor",
}

stay_intime = icustays.set_index("stay_id")["intime"].to_dict()
chunks = []

for chunk in pd.read_csv(
    csv_path(ICU_DIR, "chartevents.csv"),
    usecols=["stay_id", "itemid", "charttime", "valuenum"],
    parse_dates=["charttime"],
    chunksize=500_000,
    low_memory=False
):
    chunk = chunk[
        chunk["itemid"].isin(CHART_ITEMS) &
        chunk["valuenum"].notna() &
        chunk["stay_id"].isin(stay_intime)
    ].copy()
    chunk["intime"] = chunk["stay_id"].map(stay_intime)
    chunk = chunk[
        (chunk["charttime"] >= chunk["intime"]) &
        (chunk["charttime"] <= chunk["intime"] + pd.Timedelta(hours=24))
    ]
    chunks.append(chunk[["stay_id", "itemid", "valuenum"]])

chartevents = pd.concat(chunks, ignore_index=True)
chartevents["vital_name"] = chartevents["itemid"].map(CHART_ITEMS)

# GCS total
gcs_cols = ["gcs_eye", "gcs_verbal", "gcs_motor"]
gcs_total = (
    chartevents[chartevents["vital_name"].isin(gcs_cols)]
    .groupby(["stay_id", "vital_name"])["valuenum"].mean()
    .unstack().sum(axis=1).reset_index()
    .rename(columns={0: "gcs_total"})
)

charts = (
    chartevents.groupby(["stay_id", "vital_name"])["valuenum"]
    .mean().unstack("vital_name").reset_index()
)
charts.columns = ["stay_id"] + [f"vital_{c}" for c in charts.columns[1:]]
charts = charts.merge(gcs_total, on="stay_id", how="left")

print(f"{len(charts)} stays with vital data, {charts.shape[1]-1} vital features")
charts.head(3)

85152 stays with vital data, 11 vital features


,stay_id,vital_dbp,vital_gcs_eye,vital_gcs_motor,vital_gcs_verbal,vital_heart_rate,vital_map,vital_resp_rate,vital_sbp,vital_spo2,vital_temperature_f,gcs_total
0,30000153,72.000,3.312500,5.812500,3.187500,106.840000,86.333333,15.076923,137.111111,96.640000,99.500000,12.3125
1,30000213,56.160,3.714286,5.714286,3.571429,81.680000,79.120000,19.080000,134.920000,98.960000,98.612500,13.0000
2,30000484,55.625,3.600000,3.000000,1.000000,89.958333,65.083333,14.708333,105.750000,99.791667,96.642857,7.6000


---
## 8. Build the Cohort

Join out from ICUstay combining all dfs into our final cohort df

In [10]:
cohort = (
    icustays
    .merge(admissions,    on=["subject_id", "hadm_id"])
    .merge(patients,      on="subject_id")
    .merge(comorbidities, on="hadm_id",  how="left")
    .merge(labs,          on="stay_id",  how="left")
    .merge(charts,        on="stay_id",  how="left")
)

cm_cols = [c for c in cohort.columns if c.startswith("cm_")]
cohort[cm_cols] = cohort[cm_cols].fillna(0)

# Drop raw timestamps 
cohort = cohort.drop(columns=["intime", "outtime"])

print(f"Cohort: {len(cohort)} stays, {cohort.shape[1]} columns")
print(f"Mortality rate: {cohort['hospital_expire_flag'].mean():.2%}")

Cohort: 64957 stays, 47 columns
Mortality rate: 10.10%


---
## 9. Missing Check

Checking number of missings for the features


In [11]:
missing = cohort.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0]
missing *= 100
print(missing.to_string())

lab_lactate            47.617655
lab_ph                 45.921148
vital_map               8.105362
vital_dbp               8.089967
vital_sbp               8.080730
vital_temperature_f     6.027064
lab_hemoglobin          3.737857
lab_wbc                 3.724002
lab_platelets           3.702449
lab_glucose             3.463830
lab_hematocrit          3.457672
lab_anion_gap           3.234447
lab_bicarbonate         3.145158
lab_urea_nitrogen       3.134381
lab_potassium           3.115907
lab_creatinine          3.109750
lab_chloride            3.086657
lab_sodium              3.085118
marital_status          2.243022
vital_gcs_motor         0.535739
vital_gcs_verbal        0.520344
vital_gcs_eye           0.503410
gcs_total               0.491094
vital_resp_rate         0.326370
vital_spo2              0.237080
vital_heart_rate        0.106224
los                     0.010776


---
## 10. Quick Sanity Checks



In [12]:
print("Race group distribution:")
print(cohort["race_group"].value_counts())
print()
print("Insurance group distribution:")
print(cohort["insurance_group"].value_counts())
print()
print("Mortality by race group:")
print(cohort.groupby("race_group")["hospital_expire_flag"].agg(["mean", "count"]).round(3))

Race group distribution:
race_group
WHITE         53124
BLACK          7821
OTHER          2828
ASIAN          1005
INDIGENOUS      179
Name: count, dtype: int64

Insurance group distribution:
insurance_group
government    45472
private       19485
Name: count, dtype: int64

Mortality by race group:
             mean  count
race_group              
ASIAN       0.127   1005
BLACK       0.097   7821
INDIGENOUS  0.112    179
OTHER       0.099   2828
WHITE       0.101  53124


In [13]:
cohort.head()

,hadm_id,subject_id,stay_id,los,long_icu_stay,marital_status,hospital_expire_flag,los_hospital_hrs,is_emergency,race_group,...,vital_gcs_eye,vital_gcs_motor,vital_gcs_verbal,vital_heart_rate,vital_map,vital_resp_rate,vital_sbp,vital_spo2,vital_temperature_f,gcs_total
0,20000094,14046553,35605481,0.839664,0,WIDOWED,1,33.35,1,WHITE,...,3.333333,5.0,3.888889,110.173913,57.318182,20.863636,78.904762,98.625000,97.575000,12.222222
1,20000351,17913090,30593599,0.860544,0,MARRIED,0,135.72,1,BLACK,...,3.000000,6.0,5.000000,90.545455,99.250000,21.285714,118.000000,96.909091,98.800000,14.000000
2,20000808,16788749,35191063,0.645683,0,SINGLE,0,303.18,1,WHITE,...,4.000000,6.0,5.000000,83.066667,103.833333,12.533333,142.666667,97.600000,97.333333,15.000000
3,20000916,15583900,39464752,0.727431,0,MARRIED,1,122.55,1,WHITE,...,2.714286,5.0,3.285714,60.000000,74.727273,17.294118,121.000000,97.352941,97.620000,11.000000
4,20001305,16003661,36916968,2.782650,0,SINGLE,1,64.42,1,WHITE,...,3.000000,4.0,1.000000,77.954545,63.774194,21.354839,96.709677,94.241379,98.000000,8.000000


---
## 11. Save

Save the cohort so we don't have to re-run the loading pipeline every time.

In [14]:
cohort.to_csv("cohort.csv", index=False)
print("Saved to cohort.csv")

Saved to cohort.csv
